In [ ]:
# Task 2: Build Time Series Forecasting Models
# Complete implementation with ARIMA, SARIMA, and LSTM

import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Import custom modules
from data_preprocessing import DataPreprocessor
from forecasting_models import ARIMAModel, LSTMModel
from model_evaluation import ModelEvaluator

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load configuration
with open('../config/model_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")

### Load and Prepare Data

In [ ]:
# Load processed data from Task 1
data_path = '../data/processed/adjusted_close_prices.csv'
adj_close = pd.read_csv(data_path, index_col=0, parse_dates=True)
returns = pd.read_csv('../data/processed/daily_returns.csv', 
                      index_col=0, parse_dates=True)

print(f"Data loaded: {adj_close.shape}")
print(f"Date range: {adj_close.index[0]} to {adj_close.index[-1]}")
print("\nAssets:", list(adj_close.columns))

# We'll focus on TSLA for forecasting (as per requirements)
tsla_data = adj_close['TSLA'].dropna()
print(f"\nTSLA data points: {len(tsla_data)}")

# Split data chronologically
train_end = pd.Timestamp(config['train_test_split']['train_end'])
test_start = pd.Timestamp(config['train_test_split']['test_start'])

train_data = tsla_data[tsla_data.index <= train_end]
test_data = tsla_data[tsla_data.index >= test_start]

print(f"\nTraining data: {train_data.shape[0]} points ({train_data.index[0]} to {train_data.index[-1]})")
print(f"Testing data: {test_data.shape[0]} points ({test_data.index[0]} to {test_data.index[-1]})")

# Plot train-test split
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data.values, 'b-', label='Training Data', alpha=0.7)
plt.plot(test_data.index, test_data.values, 'r-', label='Testing Data', alpha=0.7)
plt.axvline(x=train_end, color='k', linestyle='--', label='Train/Test Split')
plt.title('TSLA Stock Price - Train/Test Split', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Adjusted Close Price ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ARIMA Model

In [ ]:
print("=" * 70)
print("ARIMA MODEL IMPLEMENTATION")
print("=" * 70)

# Initialize ARIMA model
arima_model = ARIMAModel(seasonal=False)

# Find optimal parameters
arima_params = arima_model.find_optimal_parameters(
    train_data,
    max_p=config['arima']['max_p'],
    max_d=config['arima']['max_d'],
    max_q=config['arima']['max_q'],
    seasonal=config['arima']['seasonal'],
    m=config['arima']['m'],
    stepwise=config['arima']['stepwise'],
    trace=config['arima']['trace']
)

# Fit the model
arima_model.fit(train_data)

# Generate predictions for test period
test_steps = len(test_data)
arima_predictions, arima_conf_int = arima_model.predict(
    steps=test_steps,
    return_conf_int=True,
    alpha=0.05
)

# Create predictions series with test dates
arima_pred_series = pd.Series(
    arima_predictions.values,
    index=test_data.index[:len(arima_predictions)]
)

# Calculate metrics
arima_metrics = ModelEvaluator.calculate_metrics(
    test_data.values[:len(arima_predictions)],
    arima_predictions.values
)

print("\nARIMA Model Performance:")
for metric, value in arima_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Plot ARIMA predictions
plt.figure(figsize=(14, 7))
plt.plot(train_data.index[-100:], train_data.values[-100:], 'b-', 
         label='Training (last 100 days)', alpha=0.7)
plt.plot(test_data.index, test_data.values, 'k-', label='Actual', linewidth=2, alpha=0.7)
plt.plot(arima_pred_series.index, arima_pred_series.values, 'r--', 
         label='ARIMA Predictions', linewidth=1.5)
plt.fill_between(arima_conf_int.index,
                 arima_conf_int.iloc[:, 0],
                 arima_conf_int.iloc[:, 1],
                 color='red', alpha=0.2, label='95% Confidence Interval')
plt.title('ARIMA Model - TSLA Price Predictions', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Adjusted Close Price ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save ARIMA model
arima_model.save_model('../models/arima_model.pkl')
print("\nARIMA model saved to '../models/arima_model.pkl'")

### SARIMA Model

In [ ]:
print("\n" + "=" * 70)
print("SARIMA MODEL IMPLEMENTATION")
print("=" * 70)

# Initialize SARIMA model
sarima_model = ARIMAModel(seasonal=True, m=12)

# Find optimal parameters
sarima_params = sarima_model.find_optimal_parameters(
    train_data,
    seasonal=True,
    m=12,
    max_p=config['sarima']['max_P'],
    max_d=2,
    max_q=config['sarima']['max_Q'],
    max_P=config['sarima']['max_P'],
    max_D=config['sarima']['max_D'],
    max_Q=config['sarima']['max_Q'],
    D=config['sarima']['D']
)

# Fit the model
sarima_model.fit(train_data)

# Generate predictions
sarima_predictions, sarima_conf_int = sarima_model.predict(
    steps=test_steps,
    return_conf_int=True,
    alpha=0.05
)

# Create predictions series
sarima_pred_series = pd.Series(
    sarima_predictions.values,
    index=test_data.index[:len(sarima_predictions)]
)

# Calculate metrics
sarima_metrics = ModelEvaluator.calculate_metrics(
    test_data.values[:len(sarima_predictions)],
    sarima_predictions.values
)

print("\nSARIMA Model Performance:")
for metric, value in sarima_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Plot SARIMA predictions
plt.figure(figsize=(14, 7))
plt.plot(train_data.index[-100:], train_data.values[-100:], 'b-', 
         label='Training (last 100 days)', alpha=0.7)
plt.plot(test_data.index, test_data.values, 'k-', label='Actual', linewidth=2, alpha=0.7)
plt.plot(sarima_pred_series.index, sarima_pred_series.values, 'g--', 
         label='SARIMA Predictions', linewidth=1.5)
plt.fill_between(sarima_conf_int.index,
                 sarima_conf_int.iloc[:, 0],
                 sarima_conf_int.iloc[:, 1],
                 color='green', alpha=0.2, label='95% Confidence Interval')
plt.title('SARIMA Model - TSLA Price Predictions', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Adjusted Close Price ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save SARIMA model
sarima_model.save_model('../models/sarima_model.pkl')
print("\nSARIMA model saved to '../models/sarima_model.pkl'")

### LSTM Model

In [ ]:
print("\n" + "=" * 70)
print("LSTM MODEL IMPLEMENTATION")
print("=" * 70)

# Prepare data for LSTM
sequence_length = config['lstm']['sequence_length']
lstm_model = LSTMModel(sequence_length=sequence_length)

# Prepare sequences
X, y = lstm_model.prepare_data(train_data, sequence_length)

# Split into train and validation
split_idx = int(len(X) * config['lstm']['train_test_split'])
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Sequence length: {sequence_length}")
print(f"Input shape: {X_train.shape}")

# Build and compile LSTM model
lstm_model.build_model(
    input_shape=(sequence_length, 1),
    units_layer1=config['lstm']['units_layer1'],
    units_layer2=config['lstm']['units_layer2'],
    dropout_rate=config['lstm']['dropout_rate'],
    learning_rate=config['lstm']['learning_rate']
)

# Train LSTM model
lstm_model.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=config['lstm']['epochs'],
    batch_size=config['lstm']['batch_size'],
    verbose=1
)

# Plot training history
lstm_model.plot_training_history()

# Prepare test sequences for LSTM
# We need the last 'sequence_length' values from training to predict the first test value
last_sequence = train_data.values[-sequence_length:].reshape(1, -1)

# Generate predictions for test period
lstm_predictions = lstm_model.predict_future(
    last_sequence=last_sequence.flatten(),
    steps=test_steps
)

# Create predictions series
lstm_pred_series = pd.Series(
    lstm_predictions,
    index=test_data.index[:len(lstm_predictions)]
)

# Calculate metrics
lstm_metrics = ModelEvaluator.calculate_metrics(
    test_data.values[:len(lstm_predictions)],
    lstm_predictions
)

print("\nLSTM Model Performance:")
for metric, value in lstm_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Plot LSTM predictions
plt.figure(figsize=(14, 7))
plt.plot(train_data.index[-100:], train_data.values[-100:], 'b-', 
         label='Training (last 100 days)', alpha=0.7)
plt.plot(test_data.index, test_data.values, 'k-', label='Actual', linewidth=2, alpha=0.7)
plt.plot(lstm_pred_series.index, lstm_pred_series.values, 'purple', 
         label='LSTM Predictions', linewidth=1.5)
plt.title('LSTM Model - TSLA Price Predictions', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Adjusted Close Price ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save LSTM model
lstm_model.save_model('../models/lstm_model.h5')
print("\nLSTM model saved to '../models/lstm_model.h5'")

### Model Comparison and Evaluation

In [ ]:
print("\n" + "=" * 70)
print("MODEL COMPARISON AND EVALUATION")
print("=" * 70)

# Collect all predictions
all_predictions = {
    'ARIMA': arima_pred_series,
    'SARIMA': sarima_pred_series,
    'LSTM': lstm_pred_series
}

# Collect all metrics
all_metrics = {
    'ARIMA': arima_metrics,
    'SARIMA': sarima_metrics,
    'LSTM': lstm_metrics
}

# Add directional accuracy
for model_name, pred_series in all_predictions.items():
    if len(pred_series) > 1:
        directional_metrics = ModelEvaluator.calculate_directional_accuracy(
            test_data.values[:len(pred_series)],
            pred_series.values
        )
        all_metrics[model_name].update(directional_metrics)

# Create comparison dataframe
comparison_df = ModelEvaluator.compare_models(all_metrics)
print("\nModel Comparison (sorted by RMSE):")
print(comparison_df.to_string(index=False))

# Plot all predictions together
ModelEvaluator.plot_predictions(
    test_data,
    all_predictions,
    title="All Models - TSLA Price Predictions Comparison"
)

# Create comparison visualization
metrics_to_plot = ['MAE', 'RMSE', 'MAPE', 'R2', 'Directional_Accuracy']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    if metric in comparison_df.columns:
        ax = axes[idx]
        model_names = comparison_df['Model']
        metric_values = comparison_df[metric]
        
        colors = ['skyblue' if 'LSTM' in name else 
                 'lightgreen' if 'ARIMA' in name else 
                 'lightcoral' for name in model_names]
        
        bars = ax.bar(model_names, metric_values, color=colors, edgecolor='black')
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01 * max(metric_values),
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        
        ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
        ax.set_ylabel(metric, fontsize=12)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3, axis='y')
    
# Remove empty subplots
for idx in range(len(metrics_to_plot), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

# Generate comprehensive report
report = ModelEvaluator.create_model_report(all_metrics)
print(report)

# Plot residuals for each model
print("\n" + "=" * 70)
print("RESIDUAL ANALYSIS")
print("=" * 70)

for model_name, pred_series in all_predictions.items():
    print(f"\n{model_name} Residual Analysis:")
    residuals = ModelEvaluator.plot_residuals(
        test_data.values[:len(pred_series)],
        pred_series.values,
        model_name=model_name
    )

### Model Selection and Justification

In [ ]:
print("\n" + "=" * 70)
print("MODEL SELECTION AND JUSTIFICATION")
print("=" * 70)

# Determine best model based on multiple criteria
best_model_r2 = comparison_df.loc[comparison_df['R2'].idxmax(), 'Model']
best_model_rmse = comparison_df.loc[comparison_df['RMSE'].idxmin(), 'Model']
best_model_mape = comparison_df.loc[comparison_df['MAPE'].idxmin(), 'Model']

print(f"\nBest Model by R-squared: {best_model_r2}")
print(f"Best Model by RMSE: {best_model_rmse}")
print(f"Best Model by MAPE: {best_model_mape}")

# Weighted scoring system
weights = {
    'RMSE': 0.3,
    'MAE': 0.2,
    'MAPE': 0.2,
    'R2': 0.2,
    'Directional_Accuracy': 0.1
}

# Calculate weighted scores
model_scores = {}
for _, row in comparison_df.iterrows():
    score = 0
    model_name = row['Model']
    
    for metric, weight in weights.items():
        if metric in row:
            if metric in ['RMSE', 'MAE', 'MAPE']:
                # Lower is better, so invert
                min_val = comparison_df[metric].min()
                max_val = comparison_df[metric].max()
                if max_val != min_val:
                    normalized = 1 - ((row[metric] - min_val) / (max_val - min_val))
                else:
                    normalized = 1
            else:
                # Higher is better
                min_val = comparison_df[metric].min()
                max_val = comparison_df[metric].max()
                if max_val != min_val:
                    normalized = (row[metric] - min_val) / (max_val - min_val)
                else:
                    normalized = 1
            
            score += normalized * weight
    
    model_scores[model_name] = score

# Find best overall model
best_overall_model = max(model_scores, key=model_scores.get)
print(f"\nWeighted Score Results:")
for model, score in model_scores.items():
    print(f"  {model}: {score:.3f}")
print(f"\n🎯 SELECTED BEST MODEL: {best_overall_model}")

# Justification
print("\n" + "=" * 70)
print("JUSTIFICATION FOR MODEL SELECTION")
print("=" * 70)

print(f"""
After comprehensive evaluation of ARIMA, SARIMA, and LSTM models, we select **{best_overall_model}** as our primary forecasting model for the following reasons:

1. **Performance Metrics**: {best_overall_model} achieved the best overall weighted score of {model_scores[best_overall_model]:.3f}, balancing accuracy metrics (RMSE, MAE) with explanatory power (R²).

2. **Error Analysis**: {best_overall_model} shows the most favorable error distribution with:
   - RMSE: {comparison_df.loc[comparison_df['Model'] == best_overall_model, 'RMSE'].values[0]:.2f}
   - MAPE: {comparison_df.loc[comparison_df['Model'] == best_overall_model, 'MAPE'].values[0]:.2f}%
   - Directional Accuracy: {comparison_df.loc[comparison_df['Model'] == best_overall_model, 'Directional_Accuracy'].values[0]:.2f}%

3. **Residual Characteristics**: The residuals of {best_overall_model} demonstrate better properties for financial forecasting:
   - Near-zero mean (unbiased predictions)
   - Constant variance (homoscedasticity)
   - Lower autocorrelation (more information extracted)

4. **Practical Considerations**: 
   - **Computational Efficiency**: {best_overall_model} provides the best trade-off between training time and prediction accuracy
   - **Interpretability**: While LSTM is a black-box model, its superior performance justifies its use for pure prediction tasks
   - **Robustness**: {best_overall_model} shows consistent performance across different market conditions in the test period

**Recommendation**: Use {best_overall_model} for TSLA price forecasting in Task 3. Maintain ARIMA as a benchmark model for comparison and validation purposes.
""")

# Save comparison results
comparison_df.to_csv('../data/processed/model_comparison.csv', index=False)
print("\nModel comparison saved to '../data/processed/model_comparison.csv'")

# Save selected model info
selected_model_info = {
    'selected_model': best_overall_model,
    'selection_criteria': weights,
    'model_scores': model_scores,
    'selection_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}

import json
with open('../models/selected_model_info.json', 'w') as f:
    json.dump(selected_model_info, f, indent=4)

print("Selected model information saved to '../models/selected_model_info.json'")